<a href="https://colab.research.google.com/github/sanvi-g07/ml-zoomcamp/blob/main/ml_zoomcamp_hw6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Get data

In [2]:
!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv

--2026-08-13 21:48:52--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv.1’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.07s   

2026-08-13 21:48:52 (11.2 MB/s) - ‘car_fuel_efficiency.csv.1’ saved [874188/874188]



In [3]:
df = pd.read_csv("car_fuel_efficiency.csv")
df

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369
...,...,...,...,...,...,...,...,...,...,...,...
9699,140,5.0,164.0,2981.107371,17.3,2013,Europe,Diesel,Front-wheel drive,NaN,15.101802
9700,180,NaN,154.0,2439.525729,15.0,2004,USA,Gasoline,All-wheel drive,0.0,17.962326
9701,220,2.0,138.0,2583.471318,15.1,2008,USA,Diesel,All-wheel drive,-1.0,17.186587
9702,230,4.0,177.0,2905.527390,19.4,2011,USA,Diesel,Front-wheel drive,1.0,15.331551


#Clean dataset

In [4]:
df.isnull().sum()

,0
engine_displacement,0
num_cylinders,482
horsepower,708
vehicle_weight,0
acceleration,930
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,502


In [5]:
df = df.fillna(0)

In [6]:
df.isnull().sum()

,0
engine_displacement,0
num_cylinders,0
horsepower,0
vehicle_weight,0
acceleration,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0


#Split dataset

In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [8]:
len(df_full_train), len(df_train), len(df_val), len(df_test)

(7763, 5822, 1941, 1941)

In [9]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [10]:
df_train.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,120,5.0,169.0,2966.679505,13.9,2005,USA,Gasoline,Front-wheel drive,-1.0,15.301475
1,200,3.0,143.0,2950.822121,17.1,2013,Asia,Diesel,Front-wheel drive,-1.0,15.331215
2,180,6.0,180.0,3078.221669,17.4,2007,USA,Gasoline,All-wheel drive,0.0,15.336679
3,280,5.0,174.0,2797.991793,0.0,2016,USA,Diesel,All-wheel drive,0.0,15.865850
4,250,4.0,133.0,2362.426930,16.3,2010,USA,Diesel,Front-wheel drive,-1.0,18.102203


In [11]:
y_train = df_train['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']

In [12]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [13]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'engine_displacement',
       'fuel_type=Diesel', 'fuel_type=Gasoline', 'horsepower',
       'model_year', 'num_cylinders', 'num_doors', 'origin=Asia',
       'origin=Europe', 'origin=USA', 'vehicle_weight'], dtype=object)

# Decision Tree Regressor

In [14]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [15]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- vehicle_weight <= 3022.11
|   |--- value: [16.88]
|--- vehicle_weight >  3022.11
|   |--- value: [12.94]



In [16]:
y_pred_val = dt.predict(X_val)
y_pred_val

array([16.88218854, 16.88218854, 16.88218854, ..., 12.9383797 ,
       12.9383797 , 16.88218854])

In [21]:
rmse_dt = root_mean_squared_error(y_val, y_pred_val)
rmse_dt

1.6104639028827592

# Random Forest Regressor

In [19]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=1)

In [22]:
y_pred_val = rf.predict(X_val)
y_pred_val

array([18.62889858, 15.29598647, 18.22879442, ..., 14.80283652,
       13.49358341, 15.99288211])

In [23]:
rmse_rf = root_mean_squared_error(y_val, y_pred_val)
rmse_rf

0.4599777557336149

##Tune hyperparameters

###Tune n_estimators

In [33]:
errors = []

for n_estimators in range(10, 201, 10):
  rf = RandomForestRegressor(n_estimators=n_estimators, random_state=1, n_jobs=-1)
  rf.fit(X_train, y_train)
  y_pred_val = rf.predict(X_val)
  rmse = root_mean_squared_error(y_val, y_pred_val)
  errors.append((n_estimators, round(rmse, 3)))

In [34]:
df_errors = pd.DataFrame(errors, columns=['n_estimators', 'RMSE'])
df_errors

,n_estimators,RMSE
0,10,0.460
1,20,0.454
2,30,0.451
3,40,0.448
4,50,0.446
5,60,0.445
6,70,0.445
7,80,0.445
8,90,0.445
9,100,0.444


### Tune max-depth

In [37]:
errors = []

for max_depth in [10, 15, 20, 25]:
  rf = RandomForestRegressor(n_estimators=130, max_depth=max_depth, random_state=1, n_jobs=-1)
  rf.fit(X_train, y_train)
  y_pred_val = rf.predict(X_val)
  rmse = root_mean_squared_error(y_val, y_pred_val)
  errors.append((max_depth, round(rmse, 3)))

In [38]:
df_errors = pd.DataFrame(errors, columns=['max_depth', 'RMSE'])
df_errors

,max_depth,RMSE
0,10,0.441
1,15,0.443
2,20,0.444
3,25,0.443


##Feature importance

In [40]:
# Model parameters from Question 5

rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_val = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred_val)
rmse

0.45910895978133603

In [57]:
ft_importances = pd.DataFrame(rf.feature_importances_, index=dv.get_feature_names_out(), columns=["importance"])
ft_importances.sort_values(by="importance", ascending=False)

,importance
vehicle_weight,0.959162
horsepower,0.016040
acceleration,0.011471
engine_displacement,0.003269
model_year,0.003182
num_cylinders,0.002359
num_doors,0.001591
origin=USA,0.000555
origin=Europe,0.000520
origin=Asia,0.000476


#XGBoost Model

In [62]:
features = dv.get_feature_names_out().tolist()
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [63]:
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [70]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

xgb_model = xgb.train(xgb_params, dtrain,
                      num_boost_round=100,
                      verbose_eval=5,
                      evals=watchlist)

[0]	train-rmse:1.81398	val-rmse:1.85452
[5]	train-rmse:0.51323	val-rmse:0.55158
[10]	train-rmse:0.37125	val-rmse:0.44217
[15]	train-rmse:0.34582	val-rmse:0.43578
[20]	train-rmse:0.33420	val-rmse:0.43672
[25]	train-rmse:0.32460	val-rmse:0.43774
[30]	train-rmse:0.31342	val-rmse:0.44025
[35]	train-rmse:0.30312	val-rmse:0.44158
[40]	train-rmse:0.29287	val-rmse:0.44224
[45]	train-rmse:0.28510	val-rmse:0.44368
[50]	train-rmse:0.27957	val-rmse:0.44449
[55]	train-rmse:0.27339	val-rmse:0.44541
[60]	train-rmse:0.26317	val-rmse:0.44503
[65]	train-rmse:0.25769	val-rmse:0.44644
[70]	train-rmse:0.25251	val-rmse:0.44670
[75]	train-rmse:0.24616	val-rmse:0.44733
[80]	train-rmse:0.23832	val-rmse:0.44883
[85]	train-rmse:0.23423	val-rmse:0.44992
[90]	train-rmse:0.22609	val-rmse:0.45069
[95]	train-rmse:0.21880	val-rmse:0.45167
[99]	train-rmse:0.21540	val-rmse:0.45170


In [71]:
xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

xgb_model = xgb.train(xgb_params, dtrain,
                      num_boost_round=100,
                      verbose_eval=5,
                      evals=watchlist)

[0]	train-rmse:2.28947	val-rmse:2.34567
[5]	train-rmse:1.41233	val-rmse:1.44843
[10]	train-rmse:0.90981	val-rmse:0.93980
[15]	train-rmse:0.63380	val-rmse:0.66635
[20]	train-rmse:0.49028	val-rmse:0.53129
[25]	train-rmse:0.41920	val-rmse:0.47005
[30]	train-rmse:0.38370	val-rmse:0.44371
[35]	train-rmse:0.36491	val-rmse:0.43282
[40]	train-rmse:0.35371	val-rmse:0.42861
[45]	train-rmse:0.34672	val-rmse:0.42679
[50]	train-rmse:0.34091	val-rmse:0.42575
[55]	train-rmse:0.33486	val-rmse:0.42536
[60]	train-rmse:0.33075	val-rmse:0.42561
[65]	train-rmse:0.32661	val-rmse:0.42603
[70]	train-rmse:0.32324	val-rmse:0.42672
[75]	train-rmse:0.31991	val-rmse:0.42690
[80]	train-rmse:0.31708	val-rmse:0.42718
[85]	train-rmse:0.31469	val-rmse:0.42751
[90]	train-rmse:0.31258	val-rmse:0.42800
[95]	train-rmse:0.30982	val-rmse:0.42794
[99]	train-rmse:0.30802	val-rmse:0.42833
